# 時系列予測（Time Series Prediction）

**学習目標**

1. Keras を使用して線形モデル、DNN、および CNN モデルを構築する。
2. Keras を使用して RNN および多層 RNN モデルを構築する。
3. Keras を使用して CNN と RNN を組み合わせたモデルを構築する。

このノートブックでは、これまでのノートブックの S&P 500 株価データを使用して、ディープラーニングによる時系列予測の探索を続けます。

各銘柄の過去の終値のみを使用して、翌日の株価の騰落方向を予測するさまざまなモデルを Keras でトレーニングします。

## 必要なライブラリの読み込みと環境変数の設定

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from google.cloud import bigquery
from keras.layers import (
    GRU,
    LSTM,
    RNN,
    Bidirectional,
    Conv1D,
    Dense,
    Flatten,
    MaxPool1D,
    Reshape,
)
from keras.models import Sequential
from keras.optimizers import Adam
from keras.regularizers import L1

# 見栄えの良い図をプロットするため
%matplotlib inline
mpl.rc("axes", labelsize=14)
mpl.rc("xtick", labelsize=12)
mpl.rc("ytick", labelsize=12)

# 再現性のある結果を得るため。
from numpy.random import seed

seed(1)

In [ ]:
PROJECT = !(gcloud config get-value core/project)
PROJECT = PROJECT[0]
%env PROJECT = {PROJECT}
%env BUCKET = {PROJECT}
%env REGION = "us-central1"

## 時系列データの探索

まず BigQuery から時系列データの小さなサンプルを取得し、モデリング用にデータをクリーンアップするヘルパー関数をいくつか記述します。BigQuery の `percent_change_sp500` テーブルのデータを使用します。`close_values_prior_260` 列には、任意の銘柄の過去 260 日間の終値が含まれています。

In [ ]:
%%time
bq = bigquery.Client(project=PROJECT)

bq_query = """
#standardSQL
SELECT
  symbol,
  Date,
  direction,
  close_values_prior_260
FROM
  `stock_market.eps_percent_change_sp500`
LIMIT
  100
"""

df_stock_raw = bq.query(bq_query).to_dataframe()

In [ ]:
df_stock_raw.head()

以下の `clean_data` 関数は次の 3 つの処理を行います:
 1. inf や NA の値を削除します。
 2. `Date` フィールドを解析して文字列として読み込みます。
 3. ラベル `direction` を数値に変換し、'DOWN' を 0、'STAY' を 1、'UP' を 2 にマッピングします。

In [ ]:
def clean_data(input_df):
    """Cleans data to prepare for training.

    Args:
        input_df: Pandas dataframe.
    Returns:
        Pandas dataframe.
    """
    df = input_df.copy()

    # TF は DataFrame 内の日時型（datetime）を受け付けません。
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df["Date"] = df["Date"].dt.strftime("%Y-%m-%d")

    # TF は数値ラベルを必要とします。
    df["direction_numeric"] = df["direction"].apply(
        lambda x: {"DOWN": 0, "STAY": 1, "UP": 2}[x]
    )
    return df

In [ ]:
df_stock = clean_data(df_stock_raw)

In [ ]:
df_stock.head()

## データの読み込みと前処理

モデリングを開始する前に、特徴量を z スコアにスケーリングして前処理します。これにより、モデルに投入される特徴量の値の範囲が比較可能になり、勾配降下法における収束が促進されます。

In [ ]:
STOCK_HISTORY_COLUMN = "close_values_prior_260"
COL_NAMES = ["day_" + str(day) for day in range(0, 260)]
LABEL = "direction_numeric"

In [ ]:
def _scale_features(df):
    """z-scale feature columns of Pandas dataframe.

    Args:
        features: Pandas dataframe.
    Returns:
        Pandas dataframe with each column standardized according to the
        values in that column.
    """
    avg = df.mean()
    std = df.std()
    return (df - avg) / std


def create_features(df, label_name):
    """Create modeling features and label from Pandas dataframe.

    Args:
        df: Pandas dataframe.
        label_name: str, the column name of the label.
    Returns:
        Pandas dataframe
    """
    # 終値のリストを含む 1 列を 260 列に展開します。
    time_series_features = df[STOCK_HISTORY_COLUMN].apply(pd.Series)

    # 列名を変更します。
    time_series_features.columns = COL_NAMES
    time_series_features = _scale_features(time_series_features)

    # 時系列特徴量を静的特徴量およびラベルと結合します。
    label_column = df[LABEL]

    return pd.concat([time_series_features, label_column], axis=1)

In [ ]:
df_features = create_features(df_stock, LABEL)

In [ ]:
df_features.head()

いくつかの例をプロットして、前処理のステップが正しく実装されていることを確認しましょう。

In [ ]:
ix_to_plot = [0, 1, 9, 5]
fig, ax = plt.subplots(1, 1, figsize=(15, 8))
for ix in ix_to_plot:
    label = df_features["direction_numeric"].iloc[ix]
    example = df_features[COL_NAMES].iloc[ix]
    ax = example.plot(label=label, ax=ax)
    ax.set_ylabel("scaled price")
    ax.set_xlabel("prior days")
    ax.legend()

### トレーニング/評価/テスト分割の作成

次に、トレーニング/検証/テストのデータセットに対して再現可能な分割を作成し、これらのデータセットをローカルの CSV ファイルに保存します。以下のクエリは、データセット全体のサブサンプルを取得し、トレーニング/検証/テストセットに 70-15-15 の比率で分割を作成します。

In [ ]:
def _create_split(phase):
    """Create string to produce train/valid/test splits for a SQL query.

    Args:
        phase: str, either TRAIN, VALID, or TEST.
    Returns:
        String.
    """
    floor, ceiling = "2002-11-01", "2010-07-01"
    if phase == "VALID":
        floor, ceiling = "2010-07-01", "2011-09-01"
    elif phase == "TEST":
        floor, ceiling = "2011-09-01", "2012-11-30"
    return """
    WHERE Date >= '{}'
    AND Date < '{}'
    """.format(
        floor, ceiling
    )


def create_query(phase):
    """Create SQL query to create train/valid/test splits on subsample.

    Args:
        phase: str, either TRAIN, VALID, or TEST.
        sample_size: str, amount of data to take for subsample.
    Returns:
        String.
    """
    basequery = """
    #standardSQL
    SELECT
      symbol,
      Date,
      direction,
      close_values_prior_260
    FROM
      `stock_market.eps_percent_change_sp500`
    """

    return basequery + _create_split(phase)

In [ ]:
bq = bigquery.Client(project=PROJECT)

for phase in ["TRAIN", "VALID", "TEST"]:
    # 1. クエリ文字列を作成
    query_string = create_query(phase)
    # 2. 結果を DataFrame に読み込む
    df = bq.query(query_string).to_dataframe()

    # 3. DataFrame をクリーンアップし、前処理する
    df = clean_data(df)
    df = create_features(df, label_name="direction_numeric")

    # 4. DataFrame を CSV に書き込む
    if not os.path.exists("../data"):
        os.mkdir("../data")
    df.to_csv(
        f"../data/stock-{phase.lower()}.csv",
        index_label=False,
        index=False,
    )
    print(
        "Wrote {} lines to {}".format(
            len(df), f"../data/stock-{phase.lower()}.csv"
        )
    )

In [ ]:
ls -la ../data

## モデリング

実験目的のため、上記で作成した `.csv` ファイルを使用して、メモリに収まるデータを用いてさまざまなモデルをトレーニングします。

In [ ]:
N_TIME_STEPS = 260
N_LABELS = 3

Xtrain = pd.read_csv("../data/stock-train.csv")
Xvalid = pd.read_csv("../data/stock-valid.csv")

ytrain = Xtrain.pop(LABEL)
yvalid = Xvalid.pop(LABEL)

ytrain_categorical = keras.utils.to_categorical(ytrain.values)
yvalid_categorical = keras.utils.to_categorical(yvalid.values)

トレーニングの進行状況を監視し、さまざまなモデルの評価指標を比較するために、以下の関数を使用して、トレーニングジョブからキャプチャされたトレーニング損失/検証損失や精度などの指標をプロットします。

In [ ]:
def plot_curves(train_data, val_data, label="Accuracy"):
    """Plot training and validation metrics on single axis.

    Args:
        train_data: list, metrics obtrained from training data.
        val_data: list, metrics obtained from validation data.
        label: str, title and label for plot.
    Returns:
        Matplotlib plot.
    """
    plt.plot(
        np.arange(len(train_data)) + 0.5,
        train_data,
        "b.-",
        label="Training " + label,
    )
    plt.plot(
        np.arange(len(val_data)) + 1,
        val_data,
        "r.-",
        label="Validation " + label,
    )
    plt.gca().xaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))
    plt.legend(fontsize=14)
    plt.xlabel("Epochs")
    plt.ylabel(label)
    plt.grid(True)

### ベースライン

Keras でのモデリングを開始する前に、単純なヒューリスティックを使用してベンチマークを作成しましょう。トレーニングセットの最頻クラス（多数派クラス）を予測した場合に、検証セットでどのような精度が得られるかを確認します。

In [ ]:
sum(yvalid == ytrain.value_counts().idxmax()) / yvalid.shape[0]

なるほど。最も一般的な結果である `UP` を単純に推測するだけでも、検証セットで約 29.5% の精度が得られます。

### 線形モデル

シーケンシャル（時系列）入力を単一の全結合 Dense レイヤーにマッピングするシンプルな線形モデルから始めます。

**ラボ課題 #1a:** 下のセルで、シーケンシャル入力を単一の全結合 Dense レイヤーにマッピングする [Keras Sequential API](https://keras.io/api/models/sequential/) を使用して線形モデルを作成してください。

In [ ]:
# TODO 1a
model = Sequential()

model.add(
    # TODO: ここにコードを記述してください。
)

model.compile(
    # TODO: ここにコードを記述してください。
)

history = model.fit(
    # TODO: ここにコードを記述してください。
)

In [ ]:
plot_curves(history.history["loss"], history.history["val_loss"], label="Loss")

In [ ]:
plot_curves(
    history.history["accuracy"],
    history.history["val_accuracy"],
    label="Accuracy",
)

精度はかなり早く横ばいになるようです。精度を報告するために、トレーニングの最後の数エポックにわたる検証セットでの精度を平均します。

In [ ]:
np.mean(history.history["val_accuracy"][-5:])

### ディープニューラルネットワーク（DNN）

線形モデルは、単純なベンチマークよりも改善されています。より複雑なモデルを使用すれば、さらに良い結果が得られるかもしれません。次に、Keras を使用してディープニューラルネットワーク（DNN）を作成します。ここでは 2 層の DNN を試しますが、より複雑なモデルを試したり、パフォーマンスを向上させるための追加の手法を自由に追加したりしてください。

**ラボ課題 #1b:** 下のセルで、`direction_numeric` をモデル化するためのディープニューラルネットワークを Keras で作成してください。さまざまな活性化関数を試したり、正則化を追加したりして、パフォーマンスがどれだけ向上するか確認してください。

In [ ]:
# TODO 1b
model = Sequential()

# TODO: ここにコードを記述してください。

In [ ]:
plot_curves(history.history["loss"], history.history["val_loss"], label="Loss")

In [ ]:
plot_curves(
    history.history["accuracy"],
    history.history["val_accuracy"],
    label="Accuracy",
)

In [ ]:
np.mean(history.history["val_accuracy"][-5:])

### 畳み込みニューラルネットワーク（CNN）

DNN の方がわずかに優れています。畳み込みニューラルネットワーク（CNN）のパフォーマンスがどうなるか見てみましょう。

1 次元畳み込みは、時系列データから特徴量を抽出したり、データセットのより短い固定長のセグメントから特徴量を導出したりするのに役立ちます。[Keras の Conv1D](https://keras.io/api/layers/convolution_layers/convolution1d/) の実装方法についてはドキュメントを確認してください。マックスプーリングは、畳み込みニューラルネットワークとともによく使用されるダウンサンプリング戦略です。次に、`Conv1D` を使用して畳み込みレイヤーを作成し、全結合 Dense レイヤーに渡す前に `MaxPool1D` を使用してマックスプーリングを実行する CNN モデルを Keras で構築します。

**ラボ課題 #1c:** Keras で 1 次元畳み込みネットワーク（1D CNN）を作成してください。畳み込みレイヤーの数、フィルタサイズ、カーネルサイズ、ストライド、プーリングレイヤーなどを様々に試すことができます。畳み込みレイヤーを通過させた後、結果を平坦化（Flatten）し、ディープニューラルネットワークを通過させてモデルを完成させます。

In [ ]:
# TODO 1c
model = Sequential()

# 畳み込みレイヤー
# TODO: ここにコードを記述してください。

# 結果を平坦化（Flatten）して DNN に渡します。
# TODO: ここにコードを記述してください。

# モデルをコンパイルしてトレーニングします
# TODO: ここにコードを記述してください。

In [ ]:
plot_curves(history.history["loss"], history.history["val_loss"], label="Loss")

In [ ]:
plot_curves(
    history.history["accuracy"],
    history.history["val_accuracy"],
    label="Accuracy",
)

In [ ]:
np.mean(history.history["val_accuracy"][-5:])

### リカレントニューラルネットワーク（RNN）

RNN は、時系列データの学習に特に適しています。1 つのセルの出力を次のステップの入力として渡すことで、あるイテレーションから次のイテレーションへと状態情報を保持します。下のセルでは、Keras で RNN モデルを構築します。RNN の最終状態がキャプチャされ、全結合レイヤーに渡されて予測が生成されます。

**ラボ課題 #2a:** Keras で RNN モデルを作成してください。[LSTM](https://keras.io/api/layers/recurrent_layers/lstm/)、[GRU](https://keras.io/api/layers/recurrent_layers/gru/)、または基本的な [RNN セル](https://keras.io/api/layers/recurrent_layers/simple_rnn_cell/) など、さまざまなタイプの RNN セルを試すことができます。異なるセルサイズ、活性化関数、正則化などを実験してみてください。

In [ ]:
# TODO 2a
model = Sequential()

# RNN レイヤーに渡すために入力をリシェイプします。
# TODO: ここにコードを記述してください。

# モデルをコンパイルしてトレーニングします
# TODO: ここにコードを記述してください。

In [ ]:
plot_curves(history.history["loss"], history.history["val_loss"], label="Loss")

In [ ]:
plot_curves(
    history.history["accuracy"],
    history.history["val_accuracy"],
    label="Accuracy",
)

In [ ]:
np.mean(history.history["val_accuracy"][-5:])

### 多層 RNN（Multi-layer RNN）

次に、多層 RNN を構築します。ディープニューラルネットワークの複数のレイヤーによりトレーニング中により複雑な特徴量を学習できるように、追加の RNN レイヤーによって時系列データの複雑な特徴量を学習できる可能性があります。多層 RNN では、最初の RNN レイヤーの出力が次の RNN レイヤーの入力として渡されます。

**ラボ課題 #2b:** 単層 RNN の構築方法を確認できたので、ディープな多層 RNN モデルを作成してください。RNN のレイヤーをインスタンス化する際に `return_sequences` 変数をどのように設定すべきか調べてみてください。

In [ ]:
# TODO 2b

model = Sequential()

# RNN レイヤーに渡すために入力をリシェイプします。
# TODO: ここにコードを記述してください。

# モデルをコンパイルしてトレーニングします
# TODO: ここにコードを記述してください。

In [ ]:
plot_curves(history.history["loss"], history.history["val_loss"], label="Loss")

In [ ]:
plot_curves(
    history.history["accuracy"],
    history.history["val_accuracy"],
    label="Accuracy",
)

In [ ]:
np.mean(history.history["val_accuracy"][-5:])

### CNN と RNN のアーキテクチャの組み合わせ

最後に、畳み込みネットワークとリカレントネットワークの両方の側面を組み合わせたモデルアーキテクチャを見ていきます。たとえば、1 次元畳み込みレイヤーを使用してシーケンスを処理し、特徴量を作成してから、予測前に RNN モデルに渡すことができます。

**ラボ課題 #3a:** まず 1 次元畳み込み（1D-CNN）を通過させ、次にその時系列特徴量を時系列リカレントレイヤーに渡すモデルを作成してください。CNN と RNN のさまざまなハイパーパラメータを実験して、モデルのパフォーマンスをどれだけ向上できるか確認してください。

In [ ]:
# TODO 3a
model = Sequential()

# 畳み込みレイヤーのために入力をリシェイプします
# TODO: ここにコードを記述してください。

# 畳み込み特徴量を RNN レイヤーに渡します
# TODO: ここにコードを記述してください。

# モデルをコンパイルしてトレーニングします
# TODO: ここにコードを記述してください。

In [ ]:
plot_curves(history.history["loss"], history.history["val_loss"], label="Loss")

In [ ]:
plot_curves(
    history.history["accuracy"],
    history.history["val_accuracy"],
    label="Accuracy",
)

In [ ]:
np.mean(history.history["val_accuracy"][-5:])

RNN の出力から 1 次元 CNN を使用して特徴量を作成するハイブリッドモデルの構築を試すこともできます。

**ラボ課題 #3b:** 最後に、1 次元畳み込みを適用する前に、まず RNN のリカレントレイヤーを通過させるモデルを作成してください。前回と同様に、CNN の結果は平坦化（Flatten）されて全結合レイヤーに渡されます。

In [ ]:
# TODO 3b
model = Sequential()

# 入力をリシェイプして RNN レイヤーに渡します。
# TODO: ここにコードを記述してください。

# RNN 出力に 1 次元畳み込みを適用します。
# TODO: ここにコードを記述してください。

# 畳み込み出力を平坦化（Flatten）して DNN に渡します。
# TODO: ここにコードを記述してください。

# モデルをコンパイルしてトレーニングします
# TODO: ここにコードを記述してください。

In [ ]:
plot_curves(history.history["loss"], history.history["val_loss"], label="Loss")

In [ ]:
plot_curves(
    history.history["accuracy"],
    history.history["val_accuracy"],
    label="Accuracy",
)

In [ ]:
np.mean(history.history["val_accuracy"][-5:])

## エクストラクレジット（発展課題）

1. `eps_percent_change_sp500` テーブルには、各サンプルの静的特徴量もあります。具体的には、前のラボで作成したエンジニアリングされた特徴量であり、過去 5 日、20 日、260 日間にわたる `MAX`、`MIN`、`AVG`、`STD` を取得した集計情報です（例: `close_MIN_prior_5_days`、`close_MIN_prior_20_days`、`close_MIN_prior_260_days` など）。上記で使用したシーケンス特徴量に加えて、これらの特徴量を組み込んだモデルを構築してみてください。これによりパフォーマンスは向上しますか？

2. `eps_percent_change_sp500` テーブルには、1 株当たり利益（EPS）に関する情報を取得する `surprise` 特徴量も含まれています。上記で使用したシーケンス特徴量に加えて、`surprise` 特徴量を使用するモデルを構築してみてください。これによりパフォーマンスは向上しますか？

Copyright 2025 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License